# RCP211 - Inverse Reinforcment Learning - Fayz El Razaz

## Construction de l'expert et des démonstrations

### Librairie

In [1]:
import os
import numpy as np
import gymnasium as gym

from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy

### Configuration

In [2]:
# =========================
# Configuration
# =========================

ENV_NAME = "CartPole-v1"

ROOT_DIR = os.getcwd ()
MODEL_DIR = os.path.join(ROOT_DIR, "models")
DATA_DIR = os.path.join(ROOT_DIR, "data")

EXPERT_MODEL_PATH = os.path.join(MODEL_DIR, "ppo_cartpole_expert")
DEMO_DATA_PATH = os.path.join(DATA_DIR, "expert_demonstrations.npz")

N_TRAINING_STEPS = 100_000
N_DEMO_EPISODES = 50
MIN_EXPERT_SCORE = 450

SEED = 42

### Préparation des dossiers

In [3]:
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

### Entraintement de notre expert

In [4]:
# =========================
# 1. Entraînement expert PPO
# =========================

env = gym.make(ENV_NAME)

model = PPO(
    policy="MlpPolicy",
    env=env,
    verbose=0,
    seed=SEED,
    device="cpu"
)

model.learn(total_timesteps=N_TRAINING_STEPS)

model.save(EXPERT_MODEL_PATH)

mean_reward, std_reward = evaluate_policy(
    model,
    env,
    n_eval_episodes=20,
    deterministic=True,
)

print("\n=== Évaluation de l'expert PPO ===")
print(f"Score moyen : {mean_reward:.2f} ± {std_reward:.2f}")


c:\Users\FaYzerR\Documents\Travail\Cours\CNAM\RCP211\env2\Lib\site-packages\stable_baselines3\common\evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(



=== Évaluation de l'expert PPO ===
Score moyen : 500.00 ± 0.00


### Collecte des démonstrations

In [5]:
# =========================
# 2. Collecte des démonstrations expertes
# =========================

demo_env = gym.make(ENV_NAME)

trajectories = []

episode_id = 0

while len(trajectories) < N_DEMO_EPISODES:
    obs, info = demo_env.reset(seed=SEED + episode_id)

    states = []
    actions = []
    rewards = []
    next_states = []
    dones = []

    done = False
    truncated = False
    episode_reward = 0

    while not (done or truncated):
        action, _ = model.predict(obs, deterministic=True)

        next_obs, reward, done, truncated, info = demo_env.step(action)

        states.append(obs)
        actions.append(action)
        rewards.append(reward)
        next_states.append(next_obs)
        dones.append(done or truncated)

        obs = next_obs
        episode_reward += reward

    if episode_reward >= MIN_EXPERT_SCORE:
        trajectory = {
            "states": np.array(states, dtype=np.float32),
            "actions": np.array(actions, dtype=np.int64),
            "rewards": np.array(rewards, dtype=np.float32),
            "next_states": np.array(next_states, dtype=np.float32),
            "dones": np.array(dones, dtype=bool),
            "episode_reward": episode_reward,
        }

        trajectories.append(trajectory)

        print(
            f"Trajectoire retenue {len(trajectories)}/{N_DEMO_EPISODES} "
            f"- score = {episode_reward}"
        )
    else:
        print(f"Trajectoire rejetée - score = {episode_reward}")

    episode_id += 1

Trajectoire retenue 1/50 - score = 500.0
Trajectoire retenue 2/50 - score = 500.0
Trajectoire retenue 3/50 - score = 500.0
Trajectoire retenue 4/50 - score = 500.0
Trajectoire retenue 5/50 - score = 500.0
Trajectoire retenue 6/50 - score = 500.0
Trajectoire retenue 7/50 - score = 500.0
Trajectoire retenue 8/50 - score = 500.0
Trajectoire retenue 9/50 - score = 500.0
Trajectoire retenue 10/50 - score = 500.0
Trajectoire retenue 11/50 - score = 500.0
Trajectoire retenue 12/50 - score = 500.0
Trajectoire retenue 13/50 - score = 500.0
Trajectoire retenue 14/50 - score = 500.0
Trajectoire retenue 15/50 - score = 500.0
Trajectoire retenue 16/50 - score = 500.0
Trajectoire retenue 17/50 - score = 500.0
Trajectoire retenue 18/50 - score = 500.0
Trajectoire retenue 19/50 - score = 500.0
Trajectoire retenue 20/50 - score = 500.0
Trajectoire retenue 21/50 - score = 500.0
Trajectoire retenue 22/50 - score = 500.0
Trajectoire retenue 23/50 - score = 500.0
Trajectoire retenue 24/50 - score = 500.0
T

### Construction du dataset

In [6]:
# =========================
# 3. Construction d'un dataset aplati
# =========================

all_states = np.concatenate([traj["states"] for traj in trajectories], axis=0)
all_actions = np.concatenate([traj["actions"] for traj in trajectories], axis=0)
all_rewards = np.concatenate([traj["rewards"] for traj in trajectories], axis=0)
all_next_states = np.concatenate([traj["next_states"] for traj in trajectories], axis=0)
all_dones = np.concatenate([traj["dones"] for traj in trajectories], axis=0)

episode_rewards = np.array(
    [traj["episode_reward"] for traj in trajectories],
    dtype=np.float32,
)

### Sauvegarde

In [7]:
# =========================
# 4. Sauvegarde
# =========================

np.savez(
    DEMO_DATA_PATH,
    states=all_states,
    actions=all_actions,
    rewards=all_rewards,
    next_states=all_next_states,
    dones=all_dones,
    episode_rewards=episode_rewards,
)

print("\n=== Dataset de démonstrations sauvegardé ===")
print(f"Chemin : {DEMO_DATA_PATH}")
print(f"Nombre de trajectoires : {len(trajectories)}")
print(f"Nombre total de transitions : {len(all_states)}")
print(f"Dimension des états : {all_states.shape}")
print(f"Dimension des actions : {all_actions.shape}")
print(f"Score moyen des démonstrations : {episode_rewards.mean():.2f}")


=== Dataset de démonstrations sauvegardé ===
Chemin : c:\Users\FaYzerR\Documents\Travail\Cours\CNAM\RCP211\4_Projet\data\expert_demonstrations.npz
Nombre de trajectoires : 50
Nombre total de transitions : 25000
Dimension des états : (25000, 4)
Dimension des actions : (25000,)
Score moyen des démonstrations : 500.00


## Construction du modèle de reward

### Environement & Librairies

In [8]:
import os
import numpy as np
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler

N_RANDOM_EPISODES = 500
N_EPOCHS = 1000
BATCH_SIZE = 256
LR = 1e-3
SEED = 42

os.makedirs(MODEL_DIR, exist_ok=True)

### Modèles de rewards

In [9]:
# ============================================================
# 1. Modèles de reward
# ============================================================

class LinearReward(nn.Module):
    """
    Reward linéaire :
    r(s) = w^T s + b
    """

    def __init__(self, state_dim):
        super().__init__()
        self.linear = nn.Linear(state_dim, 1)

    def forward(self, state):
        return self.linear(state)


class QuadraticReward(nn.Module):
    """
    Reward quadratique :
    r(s) = w^T [s, s^2] + b

    On enrichit les features avec les carrés des variables.
    """

    def __init__(self, state_dim):
        super().__init__()
        self.linear = nn.Linear(2 * state_dim, 1)

    def forward(self, state):
        features = torch.cat([state, state ** 2], dim=1)
        return self.linear(features)


class NeuralReward(nn.Module):
    """
    Reward non linéaire par réseau de neurones.
    """

    def __init__(self, state_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(state_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, state):
        return self.net(state)

### Classe d'entrainement

In [10]:
# ============================================================
# 2. Classe d'entraînement commune
# ============================================================

class RewardTrainer:
    def __init__(
        self,
        demo_data_path,
        n_random_episodes=1000,
        batch_size=256,
        lr=1e-3,
        n_epochs=200,
        seed=42,
    ):
        self.demo_data_path = demo_data_path
        self.n_random_episodes = n_random_episodes
        self.batch_size = batch_size
        self.lr = lr
        self.n_epochs = n_epochs
        self.seed = seed

        self.scaler = StandardScaler()

        self.X_tensor = None
        self.y_tensor = None
        self.expert_states = None
        self.random_states = None

    def load_expert_states(self):
        data = np.load(self.demo_data_path)
        self.expert_states = data["states"].astype(np.float32)

        print("États experts :", self.expert_states.shape)

    def generate_random_states(self):
        env = gym.make(ENV_NAME)

        random_states = []

        for episode in range(self.n_random_episodes):
            obs, info = env.reset(seed=self.seed + episode)
            done = False
            truncated = False

            while not (done or truncated):
                action = env.action_space.sample()
                next_obs, reward, done, truncated, info = env.step(action)

                random_states.append(obs)
                obs = next_obs

        self.random_states = np.array(random_states, dtype=np.float32)

        print("États random :", self.random_states.shape)

    def build_dataset(self):
        n = min(len(self.expert_states), len(self.random_states))

        rng = np.random.default_rng(self.seed)
        expert_idx = rng.choice(len(self.expert_states), size=n, replace=False)
        random_idx = rng.choice(len(self.random_states), size=n, replace=False)
        
        self.expert_states = self.expert_states[:n]
        self.random_states = self.random_states[:n]

        X = np.concatenate([self.expert_states, self.random_states], axis=0)

        y_expert = np.ones((n, 1), dtype=np.float32)
        y_random = np.zeros((n, 1), dtype=np.float32)
        y = np.concatenate([y_expert, y_random], axis=0)

        rng = np.random.default_rng(self.seed)
        indices = rng.permutation(len(X))

        X = X[indices]
        y = y[indices]

        X_scaled = self.scaler.fit_transform(X).astype(np.float32)

        self.X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
        self.y_tensor = torch.tensor(y, dtype=torch.float32)

    def prepare_data(self):
        self.load_expert_states()
        self.generate_random_states()
        self.build_dataset()

    def train_model(self, model, model_name):
        model.train()  # <-- ici
        criterion = nn.BCEWithLogitsLoss()
        optimizer = optim.Adam(model.parameters(), lr=self.lr)

        dataset_size = len(self.X_tensor)

        for epoch in range(self.n_epochs):
            permutation = torch.randperm(dataset_size)

            epoch_loss = 0.0

            for i in range(0, dataset_size, self.batch_size):
                batch_idx = permutation[i:i + self.batch_size]

                batch_X = self.X_tensor[batch_idx]
                batch_y = self.y_tensor[batch_idx]

                logits = model(batch_X)
                loss = criterion(logits, batch_y)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                epoch_loss += loss.item()

            if (epoch + 1) % 50 == 0:
                print(
                    f"{model_name} | Epoch {epoch + 1}/{self.n_epochs} "
                    f"- Loss : {epoch_loss:.4f}"
                )

        return model

    def evaluate_model(self, model, model_name):
        model.eval()

        with torch.no_grad():
            expert_scaled = self.scaler.transform(self.expert_states).astype(np.float32)
            random_scaled = self.scaler.transform(self.random_states).astype(np.float32)

            expert_tensor = torch.tensor(expert_scaled, dtype=torch.float32)
            random_tensor = torch.tensor(random_scaled, dtype=torch.float32)

            expert_scores = model(expert_tensor).numpy()
            random_scores = model(random_tensor).numpy()

        print(f"\n=== Évaluation {model_name} ===")
        print(f"Score moyen expert : {expert_scores.mean():.4f}")
        print(f"Score moyen random  : {random_scores.mean():.4f}")

    def save_model(self, model, model_name):
        model_path = os.path.join(MODEL_DIR, f"{model_name}.pth")
        scaler_path = os.path.join(MODEL_DIR, f"{model_name}_scaler.npz")

        torch.save(model.state_dict(), model_path)

        np.savez(
            scaler_path,
            mean=self.scaler.mean_,
            scale=self.scaler.scale_,
        )

        print(f"\nModèle sauvegardé : {model_path}")
        print(f"Scaler sauvegardé : {scaler_path}")

    def train_and_save(self, model, model_name):
        print(f"\n==============================")
        print(f"Entraînement du modèle : {model_name}")
        print(f"==============================")

        model = self.train_model(model, model_name)
        self.evaluate_model(model, model_name)
        self.save_model(model, model_name)

        return model

### Apprentissage des différentes rewards

In [11]:
# ============================================================
# 3. Exécution
# ============================================================

if __name__ == "__main__":
    trainer = RewardTrainer(
        demo_data_path=DEMO_DATA_PATH,
        n_random_episodes=N_RANDOM_EPISODES,
        batch_size=BATCH_SIZE,
        lr=LR,
        n_epochs=N_EPOCHS,
        seed=SEED,
    )

    trainer.prepare_data()

    state_dim = trainer.X_tensor.shape[1]

    models = {
        "linear_reward": LinearReward(state_dim),
        "quadratic_reward": QuadraticReward(state_dim),
        "neural_reward": NeuralReward(state_dim),
    }

    trained_models = {}

    for model_name, model in models.items():
        trained_models[model_name] = trainer.train_and_save(model, model_name)

États experts : (25000, 4)
États random : (11128, 4)

Entraînement du modèle : linear_reward
linear_reward | Epoch 50/1000 - Loss : 59.3515
linear_reward | Epoch 100/1000 - Loss : 59.2796
linear_reward | Epoch 150/1000 - Loss : 59.2589
linear_reward | Epoch 200/1000 - Loss : 59.2492
linear_reward | Epoch 250/1000 - Loss : 59.2529
linear_reward | Epoch 300/1000 - Loss : 59.2487
linear_reward | Epoch 350/1000 - Loss : 59.2474
linear_reward | Epoch 400/1000 - Loss : 59.2472
linear_reward | Epoch 450/1000 - Loss : 59.2496
linear_reward | Epoch 500/1000 - Loss : 59.2474
linear_reward | Epoch 550/1000 - Loss : 59.2471
linear_reward | Epoch 600/1000 - Loss : 59.2502
linear_reward | Epoch 650/1000 - Loss : 59.2457
linear_reward | Epoch 700/1000 - Loss : 59.2473
linear_reward | Epoch 750/1000 - Loss : 59.2472
linear_reward | Epoch 800/1000 - Loss : 59.2474
linear_reward | Epoch 850/1000 - Loss : 59.2491
linear_reward | Epoch 900/1000 - Loss : 59.2485
linear_reward | Epoch 950/1000 - Loss : 59.2

## Utilisation des rewards apprises

### Librairies & dossiers de résultats

In [12]:
import os
import numpy as np
import gymnasium as gym
import torch
import torch.nn as nn

from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy


RESULTS_DIR = os.path.join(ROOT_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

N_TRAINING_STEPS = 100_000
N_EVAL_EPISODES = 40
SEED = 42

### Création du modèle de RL

In [13]:
class LearnedRewardWrapper(gym.Wrapper):
    def __init__(self, env, reward_model, scaler_mean, scaler_scale):
        super().__init__(env)

        self.reward_model = reward_model
        self.reward_model.eval()

        self.scaler_mean = scaler_mean.astype(np.float32)
        self.scaler_scale = scaler_scale.astype(np.float32)

    def step(self, action):
        obs, true_reward, terminated, truncated, info = self.env.step(action)

        obs_scaled = (obs.astype(np.float32) - self.scaler_mean) / self.scaler_scale
        obs_tensor = torch.tensor(obs_scaled, dtype=torch.float32).unsqueeze(0)

        with torch.no_grad():
            learned_reward = self.reward_model(obs_tensor).item()

        info["true_reward"] = true_reward
        info["learned_reward"] = learned_reward

        return obs, learned_reward, terminated, truncated, info


### Chargement d'une reward

In [14]:
# ============================================================
# 3. Chargement d'une reward apprise
# ============================================================

def load_reward_model(model_name, model_class, state_dim=4):
    model_path = os.path.join(MODEL_DIR, f"{model_name}.pth")
    scaler_path = os.path.join(MODEL_DIR, f"{model_name}_scaler.npz")

    model = model_class(state_dim)
    model.load_state_dict(torch.load(model_path, map_location="cpu"))
    model.eval()

    scaler_data = np.load(scaler_path)
    scaler_mean = scaler_data["mean"]
    scaler_scale = scaler_data["scale"]

    return model, scaler_mean, scaler_scale


def make_learned_reward_env(model_name, model_class):
    base_env = gym.make(ENV_NAME)

    reward_model, scaler_mean, scaler_scale = load_reward_model(
        model_name=model_name,
        model_class=model_class,
        state_dim=4,
    )

    env = LearnedRewardWrapper(
        env=base_env,
        reward_model=reward_model,
        scaler_mean=scaler_mean,
        scaler_scale=scaler_scale,
        
    )

    return env

### Entrainement avec une reward

In [15]:
# ============================================================
# 4. Entraînement PPO avec reward apprise
# ============================================================

def train_agent_with_reward(model_name, model_class):
    print("\n============================================")
    print(f"Entraînement PPO avec reward : {model_name}")
    print("============================================")

    train_env = make_learned_reward_env(model_name, model_class)

    model = PPO(
        policy="MlpPolicy",
        env=train_env,
        verbose=0,
        seed=SEED,
        device="cpu"
    )

    model.learn(total_timesteps=N_TRAINING_STEPS)

    agent_path = os.path.join(MODEL_DIR, f"ppo_with_{model_name}")
    model.save(agent_path)

    print(f"Agent sauvegardé : {agent_path}")

    return model


### Evaluation

In [16]:
# ============================================================
# 5. Évaluation sur la vraie reward CartPole
# ============================================================

def evaluate_on_true_reward(model, model_name):
    eval_env = gym.make(ENV_NAME)

    mean_reward, std_reward = evaluate_policy(
        model,
        eval_env,
        n_eval_episodes=N_EVAL_EPISODES,
        deterministic=True,
    )

    print("\n=== Évaluation vraie reward CartPole ===")
    print(f"Reward utilisée à l'entraînement : {model_name}")
    print(f"Score moyen réel : {mean_reward:.2f} ± {std_reward:.2f}")

    return mean_reward, std_reward


### Execution finale

In [17]:
# ============================================================
# 6. Exécution complète
# ============================================================

if __name__ == "__main__":

    reward_models = {
        "linear_reward": LinearReward,
        "quadratic_reward": QuadraticReward,
        "neural_reward": NeuralReward,
    }

    results = []

    for model_name, model_class in reward_models.items():
        ppo_model = train_agent_with_reward(model_name, model_class)
        mean_reward, std_reward = evaluate_on_true_reward(ppo_model, model_name)

        results.append((model_name, mean_reward, std_reward))

    print("\n============================================")
    print("Résumé final")
    print("============================================")

    for model_name, mean_reward, std_reward in results:
        print(f"{model_name:20s} : {mean_reward:.2f} ± {std_reward:.2f}")

    results_path = os.path.join(RESULTS_DIR, "learned_reward_results.csv")

    with open(results_path, "w") as f:
        f.write("model_name,mean_true_reward,std_true_reward\n")
        for model_name, mean_reward, std_reward in results:
            f.write(f"{model_name},{mean_reward},{std_reward}\n")

    print(f"\nRésultats sauvegardés : {results_path}")


Entraînement PPO avec reward : linear_reward


C:\Users\FaYzerR\AppData\Local\Temp\ipykernel_30776\4167713982.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location

Agent sauvegardé : c:\Users\FaYzerR\Documents\Travail\Cours\CNAM\RCP211\4_Projet\models\ppo_with_linear_reward


c:\Users\FaYzerR\Documents\Travail\Cours\CNAM\RCP211\env2\Lib\site-packages\stable_baselines3\common\evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(



=== Évaluation vraie reward CartPole ===
Reward utilisée à l'entraînement : linear_reward
Score moyen réel : 27.65 ± 13.64

Entraînement PPO avec reward : quadratic_reward
Agent sauvegardé : c:\Users\FaYzerR\Documents\Travail\Cours\CNAM\RCP211\4_Projet\models\ppo_with_quadratic_reward

=== Évaluation vraie reward CartPole ===
Reward utilisée à l'entraînement : quadratic_reward
Score moyen réel : 8.95 ± 0.71

Entraînement PPO avec reward : neural_reward
Agent sauvegardé : c:\Users\FaYzerR\Documents\Travail\Cours\CNAM\RCP211\4_Projet\models\ppo_with_neural_reward

=== Évaluation vraie reward CartPole ===
Reward utilisée à l'entraînement : neural_reward
Score moyen réel : 9.38 ± 0.97

Résumé final
linear_reward        : 27.65 ± 13.64
quadratic_reward     : 8.95 ± 0.71
neural_reward        : 9.38 ± 0.97

Résultats sauvegardés : c:\Users\FaYzerR\Documents\Travail\Cours\CNAM\RCP211\4_Projet\results\learned_reward_results.csv


## Changement de paradigme - Expert vs Trajectoire générées par la politique courante

In [18]:
import os
import numpy as np
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim

from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from sklearn.preprocessing import StandardScaler


ENV_NAME = "CartPole-v1"

DATA_DIR = os.path.join(ROOT_DIR, "data")
MODEL_DIR = os.path.join(ROOT_DIR, "models")
RESULTS_DIR = os.path.join(ROOT_DIR, "results")

DEMO_DATA_PATH = os.path.join(DATA_DIR, "expert_demonstrations.npz")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

SEED = 42

N_ITERATIONS = 10
N_EXPERT_TRAJECTORIES_USED = 30
N_AGENT_TRAJECTORIES_PER_ITER = 30

REWARD_EPOCHS = 200
REWARD_LR = 1e-3
PAIR_BATCH_SIZE = 32

PPO_STEPS_PER_ITER = 50_000
N_EVAL_EPISODES = 20

REWARD_CLIP = 2.0
MARGIN = 1.0


# ============================================================
# 1. Reconstruire des trajectoires expertes depuis le fichier npz
# ============================================================

def load_expert_trajectories(path, max_trajectories=30):
    data = np.load(path)

    states = data["states"].astype(np.float32)
    dones = data["dones"].astype(bool)

    trajectories = []
    current = []

    for s, d in zip(states, dones):
        current.append(s)

        if d:
            trajectories.append(np.array(current, dtype=np.float32))
            current = []

    if len(current) > 0:
        trajectories.append(np.array(current, dtype=np.float32))

    trajectories = trajectories[:max_trajectories]

    return trajectories


# ============================================================
# 2. Reward network non borné
# ============================================================

class RewardNetwork(nn.Module):
    """
    Reward d'état non bornée.

    On évite Sigmoid/Tanh ici pour ne pas saturer trop vite.
    Le clipping est appliqué uniquement au moment de donner la reward à PPO.
    """

    def __init__(self, state_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, state):
        return self.net(state)


# ============================================================
# 3. Scaler
# ============================================================

class RewardScaler:
    def __init__(self):
        self.scaler = StandardScaler()
        self.fitted = False

    def fit(self, trajectories):
        all_states = np.concatenate(trajectories, axis=0)
        self.scaler.fit(all_states)
        self.fitted = True

    def transform_states(self, states):
        return self.scaler.transform(states).astype(np.float32)


# ============================================================
# 4. Retour prédit d'une trajectoire
# ============================================================

def trajectory_return(reward_model, scaler, trajectory):
    states_scaled = scaler.transform_states(trajectory)
    states_tensor = torch.tensor(states_scaled, dtype=torch.float32)

    rewards = reward_model(states_tensor)

    # Moyenne plutôt que somme pour éviter que le modèle apprenne seulement :
    # "plus long = meilleur".
    return rewards.mean()


# ============================================================
# 5. Apprentissage reward par préférence expert > agent
# ============================================================

def train_reward_by_preferences(
    reward_model,
    scaler,
    expert_trajectories,
    agent_trajectories,
    n_epochs=200,
    lr=1e-3,
    batch_size=32,
    margin=1.0,
):
    optimizer = optim.Adam(
        reward_model.parameters(),
        lr=lr,
        weight_decay=1e-4,
    )

    reward_model.train()

    n_expert = len(expert_trajectories)
    n_agent = len(agent_trajectories)

    for epoch in range(n_epochs):
        epoch_loss = 0.0

        for _ in range(batch_size):
            expert_traj = expert_trajectories[np.random.randint(n_expert)]
            agent_traj = agent_trajectories[np.random.randint(n_agent)]

            expert_return = trajectory_return(
                reward_model,
                scaler,
                expert_traj,
            )

            agent_return = trajectory_return(
                reward_model,
                scaler,
                agent_traj,
            )

            # Ranking loss :
            # on veut expert_return >= agent_return + margin
            ranking_loss = torch.relu(
                margin - (expert_return - agent_return)
            )

            # Régularisation douce :
            # évite que la reward parte à des valeurs énormes.
            reg_loss = 1e-3 * (
                expert_return.pow(2) + agent_return.pow(2)
            )

            loss = ranking_loss + reg_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        if (epoch + 1) % 50 == 0:
            print(
                f"Reward preference epoch {epoch + 1}/{n_epochs} "
                f"- loss={epoch_loss:.4f}"
            )

    reward_model.eval()


# ============================================================
# 6. Wrapper PPO
# ============================================================

class LearnedRewardWrapper(gym.Wrapper):
    def __init__(self, env, reward_model, scaler):
        super().__init__(env)
        self.reward_model = reward_model
        self.scaler = scaler

    def step(self, action):
        obs, native_reward, terminated, truncated, info = self.env.step(action)

        obs_array = np.asarray(obs, dtype=np.float32).reshape(1, -1)
        obs_scaled = self.scaler.transform_states(obs_array)

        obs_tensor = torch.tensor(obs_scaled, dtype=torch.float32)

        self.reward_model.eval()

        with torch.no_grad():
            learned_reward = self.reward_model(obs_tensor).item()

        learned_reward = float(
            np.clip(learned_reward, -REWARD_CLIP, REWARD_CLIP)
        )

        info["native_reward_ignored"] = native_reward
        info["learned_reward"] = learned_reward

        return obs, learned_reward, terminated, truncated, info


# ============================================================
# 7. Collecte trajectoires agent
# ============================================================

def collect_agent_trajectories(model, n_episodes, seed=42, deterministic=False):
    env = gym.make(ENV_NAME)

    trajectories = []

    for episode in range(n_episodes):
        obs, info = env.reset(seed=seed + episode)

        states = []

        terminated = False
        truncated = False

        while not (terminated or truncated):
            states.append(obs)

            action, _ = model.predict(obs, deterministic=deterministic)
            action = int(action)

            obs, _, terminated, truncated, info = env.step(action)

        trajectories.append(np.array(states, dtype=np.float32))

    env.close()

    return trajectories


# ============================================================
# 8. Évaluation
# ============================================================

def evaluate_true_reward_for_analysis_only(model):
    env = Monitor(gym.make(ENV_NAME))

    mean_reward, std_reward = evaluate_policy(
        model,
        env,
        n_eval_episodes=N_EVAL_EPISODES,
        deterministic=True,
    )

    env.close()

    return mean_reward, std_reward


def evaluate_lengths(model, n_episodes=20, seed=42):
    trajectories = collect_agent_trajectories(
        model,
        n_episodes=n_episodes,
        seed=seed,
        deterministic=True,
    )

    lengths = np.array([len(t) for t in trajectories])

    return float(lengths.mean()), float(lengths.std())


def diagnostic_returns(reward_model, scaler, expert_trajectories, agent_trajectories):
    reward_model.eval()

    with torch.no_grad():
        expert_returns = [
            trajectory_return(reward_model, scaler, t).item()
            for t in expert_trajectories
        ]

        agent_returns = [
            trajectory_return(reward_model, scaler, t).item()
            for t in agent_trajectories
        ]

    print("\nDiagnostic reward preference")
    print(
        f"Return expert moyen : "
        f"{np.mean(expert_returns):.4f} ± {np.std(expert_returns):.4f}"
    )
    print(
        f"Return agent moyen  : "
        f"{np.mean(agent_returns):.4f} ± {np.std(agent_returns):.4f}"
    )


# ============================================================
# 9. Boucle principale
# ============================================================

if __name__ == "__main__":

    np.random.seed(SEED)
    torch.manual_seed(SEED)

    expert_trajectories = load_expert_trajectories(
        DEMO_DATA_PATH,
        max_trajectories=N_EXPERT_TRAJECTORIES_USED,
    )

    print(f"Nombre trajectoires expertes : {len(expert_trajectories)}")
    print(
        "Longueur moyenne expert :",
        np.mean([len(t) for t in expert_trajectories]),
    )

    state_dim = expert_trajectories[0].shape[1]

    # Au départ, on fitte le scaler sur les trajectoires expertes.
    scaler = RewardScaler()
    scaler.fit(expert_trajectories)

    reward_model = RewardNetwork(state_dim)

    train_env = gym.make(ENV_NAME)
    train_env = LearnedRewardWrapper(train_env, reward_model, scaler)

    ppo_model = PPO(
        policy="MlpPolicy",
        env=train_env,
        verbose=0,
        seed=SEED,
        device = "cpu"
    )

    # On initialise l'agent avec une politique non entraînée.
    agent_trajectories = collect_agent_trajectories(
        ppo_model,
        n_episodes=N_AGENT_TRAJECTORIES_PER_ITER,
        seed=SEED,
        deterministic=False,
    )

    results = []

    for iteration in range(N_ITERATIONS):
        print("\n============================================")
        print(f"Iteration {iteration + 1}/{N_ITERATIONS}")
        print("============================================")

        # Refit scaler sur expert + agent courant
        scaler.fit(expert_trajectories + agent_trajectories)

        # 1. Apprendre / raffiner la reward :
        #    expert trajectories > agent trajectories
        train_reward_by_preferences(
            reward_model=reward_model,
            scaler=scaler,
            expert_trajectories=expert_trajectories,
            agent_trajectories=agent_trajectories,
            n_epochs=REWARD_EPOCHS,
            lr=REWARD_LR,
            batch_size=PAIR_BATCH_SIZE,
            margin=MARGIN,
        )

        diagnostic_returns(
            reward_model,
            scaler,
            expert_trajectories,
            agent_trajectories,
        )

        # 2. Entraîner PPO avec la reward courante
        ppo_model.learn(
            total_timesteps=PPO_STEPS_PER_ITER,
            reset_num_timesteps=False,
        )

        # 3. Collecter nouvelles trajectoires de l'agent
        agent_trajectories = collect_agent_trajectories(
            ppo_model,
            n_episodes=N_AGENT_TRAJECTORIES_PER_ITER,
            seed=SEED + 1000 * iteration,
            deterministic=False,
        )

        mean_len, std_len = evaluate_lengths(
            ppo_model,
            n_episodes=N_EVAL_EPISODES,
            seed=SEED + 10_000 * iteration,
        )

        true_mean, true_std = evaluate_true_reward_for_analysis_only(ppo_model)

        print("\nÉvaluation comportementale")
        print(f"Longueur moyenne : {mean_len:.2f} ± {std_len:.2f}")
        print("\nÉvaluation vraie reward CartPole, analyse uniquement")
        print(f"Score vrai : {true_mean:.2f} ± {true_std:.2f}")

        results.append({
            "iteration": iteration + 1,
            "mean_length": mean_len,
            "std_length": std_len,
            "true_reward_mean_analysis_only": true_mean,
            "true_reward_std_analysis_only": true_std,
        })

    # Sauvegarde
    ppo_path = os.path.join(MODEL_DIR, "ppo_preference_reward")
    reward_path = os.path.join(MODEL_DIR, "reward_preference_model.pth")

    ppo_model.save(ppo_path)
    torch.save(reward_model.state_dict(), reward_path)

    results_path = os.path.join(
        RESULTS_DIR,
        "preference_reward_results.csv",
    )

    with open(results_path, "w") as f:
        f.write(
            "iteration,mean_length,std_length,"
            "true_reward_mean_analysis_only,true_reward_std_analysis_only\n"
        )

        for row in results:
            f.write(
                f"{row['iteration']},"
                f"{row['mean_length']},"
                f"{row['std_length']},"
                f"{row['true_reward_mean_analysis_only']},"
                f"{row['true_reward_std_analysis_only']}\n"
            )

    print("\nSauvegardes terminées.")
    print("PPO :", ppo_path)
    print("Reward :", reward_path)
    print("Résultats :", results_path)

Nombre trajectoires expertes : 30
Longueur moyenne expert : 500.0

Iteration 1/10
Reward preference epoch 50/200 - loss=0.0921
Reward preference epoch 100/200 - loss=0.0614
Reward preference epoch 150/200 - loss=0.1587
Reward preference epoch 200/200 - loss=0.0844

Diagnostic reward preference
Return expert moyen : 1.1503 ± 0.0281
Return agent moyen  : -1.0475 ± 0.4668

Évaluation comportementale
Longueur moyenne : 414.60 ± 155.34

Évaluation vraie reward CartPole, analyse uniquement
Score vrai : 391.00 ± 176.45

Iteration 2/10
Reward preference epoch 50/200 - loss=0.1068
Reward preference epoch 100/200 - loss=0.2015
Reward preference epoch 150/200 - loss=0.1036
Reward preference epoch 200/200 - loss=0.1085

Diagnostic reward preference
Return expert moyen : 1.1265 ± 0.0693
Return agent moyen  : -0.8987 ± 0.6203

Évaluation comportementale
Longueur moyenne : 500.00 ± 0.00

Évaluation vraie reward CartPole, analyse uniquement
Score vrai : 500.00 ± 0.00

Iteration 3/10
Reward preference 

In [19]:
scaler_path = os.path.join(MODEL_DIR, "reward_preference_scaler.npz")

np.savez(
    scaler_path,
    mean=scaler.scaler.mean_,
    scale=scaler.scaler.scale_,
)

print("Scaler :", scaler_path)

Scaler : c:\Users\FaYzerR\Documents\Travail\Cours\CNAM\RCP211\4_Projet\models\reward_preference_scaler.npz


In [20]:
# train_ppo_from_loaded_reward.py

import os
import numpy as np
import gymnasium as gym
import torch
import torch.nn as nn

from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor


# ============================================================
# Configuration
# ============================================================

ENV_NAME = "CartPole-v1"

ROOT_DIR = os.getcwd ()
MODEL_DIR = os.path.join(ROOT_DIR, "models")
RESULTS_DIR = os.path.join(ROOT_DIR, "results")

REWARD_MODEL_PATH = os.path.join(MODEL_DIR, "reward_preference_model.pth")
SCALER_PATH = os.path.join(MODEL_DIR, "reward_preference_scaler.npz")

FINAL_AGENT_PATH = os.path.join(MODEL_DIR, "ppo_from_loaded_preference_reward")

N_TRAINING_STEPS = 500_000
N_EVAL_EPISODES = 20

SEED = 42
REWARD_CLIP = 2.0

os.makedirs(RESULTS_DIR, exist_ok=True)


# ============================================================
# 1. Architecture identique au modèle de reward appris
# ============================================================

class RewardNetwork(nn.Module):
    def __init__(self, state_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, state):
        return self.net(state)


# ============================================================
# 2. Scaler rechargé
# ============================================================

class LoadedScaler:
    def __init__(self, scaler_path):
        data = np.load(scaler_path)

        self.mean = data["mean"].astype(np.float32)
        self.scale = data["scale"].astype(np.float32)

    def transform_states(self, states):
        states = np.asarray(states, dtype=np.float32)
        return ((states - self.mean) / self.scale).astype(np.float32)


# ============================================================
# 3. Chargement du modèle de reward
# ============================================================

def load_reward_model(model_path, state_dim=4):
    reward_model = RewardNetwork(state_dim)

    reward_model.load_state_dict(
        torch.load(model_path, map_location="cpu")
    )

    reward_model.eval()

    return reward_model


# ============================================================
# 4. Wrapper CartPole avec reward apprise
# ============================================================

class LoadedRewardWrapper(gym.Wrapper):
    def __init__(self, env, reward_model, scaler, reward_clip=2.0):
        super().__init__(env)

        self.reward_model = reward_model
        self.scaler = scaler
        self.reward_clip = reward_clip

    def step(self, action):
        obs, native_reward, terminated, truncated, info = self.env.step(action)

        obs_array = np.asarray(obs, dtype=np.float32).reshape(1, -1)
        obs_scaled = self.scaler.transform_states(obs_array)

        obs_tensor = torch.tensor(obs_scaled, dtype=torch.float32)

        self.reward_model.eval()

        with torch.no_grad():
            learned_reward = self.reward_model(obs_tensor).item()

        learned_reward = float(
            np.clip(learned_reward, -self.reward_clip, self.reward_clip)
        )

        info["native_reward_ignored"] = native_reward
        info["learned_reward"] = learned_reward

        return obs, learned_reward, terminated, truncated, info


# ============================================================
# 5. Création environnement reward apprise
# ============================================================

def make_loaded_reward_env():
    reward_model = load_reward_model(
        model_path=REWARD_MODEL_PATH,
        state_dim=4,
    )

    scaler = LoadedScaler(SCALER_PATH)

    env = gym.make(ENV_NAME)
    env = LoadedRewardWrapper(
        env=env,
        reward_model=reward_model,
        scaler=scaler,
        reward_clip=REWARD_CLIP,
    )

    return env


# ============================================================
# 6. Évaluation vraie reward CartPole
# ============================================================

def evaluate_on_true_cartpole_reward(model):
    eval_env = Monitor(gym.make(ENV_NAME))

    mean_reward, std_reward = evaluate_policy(
        model,
        eval_env,
        n_eval_episodes=N_EVAL_EPISODES,
        deterministic=True,
    )

    eval_env.close()

    return mean_reward, std_reward


# ============================================================
# 7. Exécution
# ============================================================

if __name__ == "__main__":

    train_env = make_loaded_reward_env()

    model = PPO(
        policy="MlpPolicy",
        env=train_env,
        verbose=0,
        seed=SEED,
        device="cpu",
    )

    print("\nEntraînement d'un nouveau PPO avec la reward apprise rechargée...")
    model.learn(total_timesteps=N_TRAINING_STEPS)

    model.save(FINAL_AGENT_PATH)

    mean_reward, std_reward = evaluate_on_true_cartpole_reward(model)

    print("\n============================================")
    print("Évaluation finale sur vraie reward CartPole")
    print("============================================")
    print(f"Score moyen réel : {mean_reward:.2f} ± {std_reward:.2f}")
    print(f"Agent sauvegardé : {FINAL_AGENT_PATH}")

    results_path = os.path.join(
        RESULTS_DIR,
        "ppo_from_loaded_preference_reward_results.csv",
    )

    with open(results_path, "w") as f:
        f.write("mean_true_reward,std_true_reward\n")
        f.write(f"{mean_reward},{std_reward}\n")

    print(f"Résultats sauvegardés : {results_path}")


Entraînement d'un nouveau PPO avec la reward apprise rechargée...


C:\Users\FaYzerR\AppData\Local\Temp\ipykernel_30776\4635164.py:82: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(model_path, map_location="cpu")



Évaluation finale sur vraie reward CartPole
Score moyen réel : 499.80 ± 0.87
Agent sauvegardé : c:\Users\FaYzerR\Documents\Travail\Cours\CNAM\RCP211\4_Projet\models\ppo_from_loaded_preference_reward
Résultats sauvegardés : c:\Users\FaYzerR\Documents\Travail\Cours\CNAM\RCP211\4_Projet\results\ppo_from_loaded_preference_reward_results.csv


In [21]:
import os
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = "results"
FIGURES_DIR = "figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

# ============================================================
# 1. Résultats PPO entraîné avec rewards apprises
# ============================================================

df_rewards = pd.read_csv(os.path.join(RESULTS_DIR, "learned_reward_results.csv"))

plt.figure(figsize=(8, 5))
plt.bar(df_rewards["model_name"], df_rewards["mean_true_reward"], yerr=df_rewards["std_true_reward"], capsize=5)
plt.ylabel("Récompense vraie moyenne")
plt.xlabel("Modèle de récompense")
plt.title("Performance des agents PPO selon la récompense apprise")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "ppo_learned_rewards.png"), dpi=300)
plt.close()


# ============================================================
# 2. Évolution des démonstrations générées par préférence
# ============================================================

df_pref = pd.read_csv(os.path.join(RESULTS_DIR, "preference_reward_results.csv"))

plt.figure(figsize=(8, 5))
plt.errorbar(
    df_pref["iteration"],
    df_pref["mean_length"],
    yerr=df_pref["std_length"],
    marker="o",
    capsize=4
)
plt.xlabel("Itération")
plt.ylabel("Longueur moyenne des trajectoires")
plt.title("Évolution de la longueur des trajectoires préférées")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "preference_trajectory_lengths.png"), dpi=300)
plt.close()


# ============================================================
# 3. Récompense vraie moyenne des trajectoires préférées
# ============================================================

plt.figure(figsize=(8, 5))
plt.errorbar(
    df_pref["iteration"],
    df_pref["true_reward_mean_analysis_only"],
    yerr=df_pref["true_reward_std_analysis_only"],
    marker="o",
    capsize=4
)
plt.xlabel("Itération")
plt.ylabel("Récompense vraie moyenne")
plt.title("Évolution de la qualité des trajectoires préférées")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "preference_true_reward_evolution.png"), dpi=300)
plt.close()


# ============================================================
# 4. PPO entraîné depuis la reward de préférence sauvegardée
# ============================================================

df_loaded = pd.read_csv(os.path.join(RESULTS_DIR, "ppo_from_loaded_preference_reward_results.csv"))

mean_reward = df_loaded.loc[0, "mean_true_reward"]
std_reward = df_loaded.loc[0, "std_true_reward"]

plt.figure(figsize=(5, 5))
plt.bar(["Reward de préférence"], [mean_reward], yerr=[std_reward], capsize=5)
plt.ylabel("Récompense vraie moyenne")
plt.title("Performance PPO avec reward de préférence sauvegardée")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "ppo_loaded_preference_reward.png"), dpi=300)
plt.close()


print("Graphiques sauvegardés dans le dossier:", FIGURES_DIR)

Graphiques sauvegardés dans le dossier: figures
